# Shadow-Net · SOREL 7M — Entrenamiento MLP (notebook canonico, v5)

En este notebook entreno el MLP de Shadow-Net Defender sobre 7,000,000 muestras de SOREL-20M
con el contrato de features `ShadowNetFeatures_v1.1 = EMBER_2381 + OVERLAY_6 = 2387`. Mantengo
intactas las Fases 0-4 ya cerradas (acceso Range/ZIP64, manifest 7M seed 42, StandardScaler
EMBER de 2381 y contrato OVERLAY_6). El trabajo de esta version es exclusivamente la Fase 5:

- Hago que el entorno sea compatible con la GPU que Kaggle me asigne (Tesla P100 `sm_60` o
  NVIDIA T4 / T4 x2 `sm_75`), instalando un PyTorch que traiga kernels para `sm_60` cuando haga falta.
- Entreno 2 epocas completas con la misma arquitectura y los mismos hiperparametros aprobados.
- Registro metricas completas por epoca (train y validacion) y genero 10 graficas como evidencia.
- Guardo checkpoints `best` y `final` limpios, ONNX-ready, sin tocar el modelo de produccion (2381).

Contrato inviolable: `X = [EMBER_2381 | OVERLAY_6]`. Nunca reordeno, elimino ni cambio la
semantica de las 2381 features EMBER; el overlay es un apendice de 6 columnas.


In [ ]:
# FASE 0.5 — Compatibilidad de hardware (se ejecuta ANTES de importar torch).
# Por que existe: el torch 2.10+cu128 de la imagen de Kaggle ya no compila kernels para
# sm_60, asi que una Tesla P100 (capability 6.0) falla con "no kernel image is available".
# Si detecto una P100, instalo torch 2.4.1 (build cu121 en PyPI: incluye sm_50..sm_90 y es
# compatible con numpy 2.x) antes de importarlo. Si Kaggle asigna T4/T4x2 (sm_75) el torch
# actual funciona y no toco nada. Si no hay GPU, se usara CPU como fallback.
import subprocess, sys, os

def _smi(query):
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=" + query, "--format=csv,noheader"],
            text=True, timeout=30).strip()
    except Exception:
        return ""

SMI_NAMES = [l.strip() for l in _smi("name").splitlines() if l.strip()]
SMI_CAPS = [l.strip() for l in _smi("compute_cap").splitlines() if l.strip()]
print("GPUs detectadas por nvidia-smi:", list(zip(SMI_NAMES, SMI_CAPS)) or "ninguna")

# P100 = nombre con "P100" o capability 6.x. Si el query de compute_cap no esta soportado,
# cae al nombre.
P100 = False
for i, n in enumerate(SMI_NAMES):
    cap = SMI_CAPS[i] if i < len(SMI_CAPS) else ""
    if "P100" in n or cap.startswith("6."):
        P100 = True
print(f"P100 detectada: {P100}")

def _installed_torch():
    try:
        from importlib.metadata import version
        return version("torch")
    except Exception:
        return None

TORCH_PRELOADED = "torch" in sys.modules
INSTALLED_TORCH = _installed_torch()
print(f"torch instalado (metadata): {INSTALLED_TORCH} | preimportado: {TORCH_PRELOADED}")

if P100 and not TORCH_PRELOADED and (INSTALLED_TORCH is None or not INSTALLED_TORCH.startswith("2.4.1")):
    print("Tesla P100 (sm_60): instalo torch==2.4.1 (cu121, con kernels sm_60)...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "torch", "torchvision", "torchaudio"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "torch==2.4.1"],
                   check=False)
    # Verifico en un subproceso para no importar torch en este kernel todavia.
    _verify = (
        "import torch;"
        "a=torch.cuda.get_arch_list();print('arch_list=',a);"
        "assert 'sm_60' in a, 'sin sm_60';"
        "x=torch.ones(64,64,device='cuda');"
        "s=float((x@x).sum().cpu());n=float(x.ne(0).float().sum().cpu());"
        "print('smoke P100 OK', s, n)"
    )
    r = subprocess.run([sys.executable, "-c", _verify], capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0:
        print("AVISO: la verificacion de P100 fallo; Fase 5 hara fallback a CPU si CUDA no sirve.")
        print(r.stderr.strip()[-600:])
    else:
        print("PASS Fase 0.5: torch compatible con sm_60 instalado y verificado.")
elif P100 and TORCH_PRELOADED:
    print("AVISO: torch ya estaba importado antes de esta celda. Si es la build cu128 sin sm_60,")
    print("       reinicio el kernel y ejecuto de nuevo para que tome la version instalada.")
else:
    print("PASS Fase 0.5: hardware no requiere cambio de torch (T4 / T4x2 / CPU).")


In [ ]:
# FASE 0 — Environment / reproducibilidad (solo observa y reporta).
import os, sys, shutil, subprocess, platform
GLOBAL_SEED = 42
import numpy as np
np.random.seed(GLOBAL_SEED)

def sh(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, timeout=15).strip()
    except Exception as e:
        return f"NA ({e})"

print("python:", sys.version.split()[0], "|", platform.platform())
print("numpy:", np.__version__)
try:
    import sklearn
    print("sklearn:", sklearn.__version__)
except ImportError:
    print("sklearn: NO INSTALADO")
try:
    import pandas
    print("pandas:", pandas.__version__)
except ImportError:
    print("pandas: NO INSTALADO")
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), end="")
    if torch.cuda.is_available():
        print(f" | {torch.cuda.get_device_name(0)} | VRAM={torch.cuda.get_device_properties(0).total_memory/2**30:.1f}GB")
    else:
        print(" | CPU-only en esta fase (scaler es CPU)")
except ImportError:
    print("torch: no instalado (se necesitara desde Fase 5)")
print("nvidia-smi:", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>&1 | head -5"))
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"RAM total={vm.total/2**30:.1f}GB disponible={vm.available/2**30:.1f}GB")
except ImportError:
    print("psutil: no instalado")
print("df /kaggle/working:", sh("df -h /kaggle/working 2>&1 | tail -1"))
print("df /tmp:", sh("df -h /tmp 2>&1 | tail -1"))
print(f"GLOBAL_SEED={GLOBAL_SEED}")
print("PASS Fase 0: entorno caracterizado.")


In [ ]:
# FASE 1 — Data access: Range/ZIP64 sobre train-features.npz (sin descargar 121GB).
# Reutiliza la logica validada H1-H4: EOCD clasico -> ZIP64 -> Central Directory ->
# Local File Header (STORED=0) -> header .npy -> PAY0/payload. Termina con asserts duros.
import os, struct, time, urllib.request, urllib.error
import numpy as np

NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"
TRAIN_SPLIT = 1543542570.0  # config.py oficial sophos/SOREL-20M
VAL_SPLIT = 1547279640.0
EXPECTED_NPZ_ROWS = 12699013
EXPECTED_NPZ_COLS = 2381
N_FEATURES = 2381
ROW_BYTES = 2381 * 4  # float32 STORED sin compresion
META_URL = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/processed-data/meta.db"
META_SIZE = 3788979200
MISSING_URL = "https://github.com/sophos/SOREL-20M/raw/master/shas_missing_ember_features.json"
SKIP_VALIDATED = True
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
META = os.path.join(WORK, "meta.db")
MISS = os.path.join(WORK, "shas_missing_ember_features.json")
PACE = 0.35

def npz_range(a, b, tries=8, timeout=120):
    # GET con reintentos: tolera 416/503/429/500 (throttling S3), aborta ante otros.
    for k in range(tries):
        try:
            time.sleep(PACE if k == 0 else 8 * k)
            req = urllib.request.Request(NPZ, headers={"Range": f"bytes={a}-{b}"})
            r = urllib.request.urlopen(req, timeout=timeout)
            assert r.status == 206, f"Range no honrado: {r.status}"
            d = r.read()
            assert len(d) == b - a + 1, f"truncado: {len(d)} vs {b-a+1}"
            return d
        except urllib.error.HTTPError as e:
            if e.code in (416, 503, 429, 500):
                continue
            raise
    raise RuntimeError(f"npz_range agotado {a}-{b}")

TOTAL = 121046992510  # verificado H2 via Content-Range
tail = npz_range(TOTAL - 131072, TOTAL - 1)
eocd = tail.rfind(b"PK\x05\x06")
assert eocd != -1, "FAIL: EOCD no encontrado"
(n_disk, n_cd, n_entries, n_total, cd_size, cd_off, _) = struct.unpack("<HHHHIIH", tail[eocd+4:eocd+22])
if cd_off == 0xFFFFFFFF:  # ZIP64: offset real de 64 bits
    loc = tail.rfind(b"PK\x06\x07")
    assert loc != -1, "FAIL: sin locator ZIP64"
    (_, zoff, _) = struct.unpack("<IQI", tail[loc+4:loc+20])
    z = npz_range(zoff, zoff + 55)
    assert z[:4] == b"PK\x06\x06", "FAIL: firma ZIP64 EOCD ausente"
    (_, _, _, _, _, _, n_total, cd_size, cd_off) = struct.unpack("<QHHIIQQQQ", z[4:56])
assert n_total >= 1 and cd_size > 0, "FAIL: inventario vacio"
cd_blob = npz_range(cd_off, cd_off + cd_size - 1)
entries, pos = [], 0
while pos < len(cd_blob):
    assert cd_blob[pos:pos+4] == b"PK\x01\x02", f"FAIL: firma CD corrupta en {pos}"
    (vneed, flags, comp, mt, md, crc, csz32, usz32, nlen, elen, clen, disk, iattr, eattr, lho32) = struct.unpack("<HHHHHIIIHHHHHII", cd_blob[pos+6:pos+46])
    name = cd_blob[pos+46:pos+46+nlen].decode()
    extra = cd_blob[pos+46+nlen:pos+46+nlen+elen]
    csize, usize, lho = csz32, usz32, lho32
    j = 0  # extra ZIP64 id 0x0001 rellena campos 0xFFFFFFFF en orden usize/csize/lho
    while j + 4 <= len(extra):
        hid, dsz = struct.unpack("<HH", extra[j:j+4])
        if hid == 0x0001:
            nq = dsz // 8
            vals = struct.unpack("<" + "Q" * nq, extra[j+4:j+4+8*nq])
            vi = 0
            if usize == 0xFFFFFFFF: usize = vals[vi]; vi += 1
            if csize == 0xFFFFFFFF: csize = vals[vi]; vi += 1
            if lho == 0xFFFFFFFF: lho = vals[vi]; vi += 1
            break
        j += 4 + dsz
    entries.append({"name": name, "compress": comp, "csize": csize, "usize": usize, "lho": lho})
    pos += 46 + nlen + elen + clen
arr = [e for e in entries if e["name"].endswith(".npy")]
assert arr, "FAIL: sin entradas .npy"
assert all(e["compress"] == 0 for e in arr), "FAIL: DEFLATE — row-Range imposible"
lho = arr[0]["lho"]
lh = npz_range(lho, lho + 29)
assert lh[:4] == b"PK\x03\x04", "FAIL: firma Local File Header ausente"
assert struct.unpack("<H", lh[8:10])[0] == 0, "FAIL: arr_0 no STORED"
nlen = struct.unpack("<H", lh[26:28])[0]
elen = struct.unpack("<H", lh[28:30])[0]
data_off = lho + 30 + nlen + elen
npy_head = npz_range(data_off, data_off + 127)
assert npy_head[:6] == b"\x93NUMPY", "FAIL: magia NPY ausente"
hlen = struct.unpack("<H", npy_head[8:10])[0]
hdr = npz_range(data_off, data_off + 10 + hlen - 1)[10:].decode("latin1")
d = eval(hdr)  # formato controlado por numpy; magia ya validada
shape, DT0, order = tuple(d["shape"]), np.dtype(d["descr"]), d["fortran_order"]
assert order is False, "FAIL: array Fortran"
assert shape == (EXPECTED_NPZ_ROWS, EXPECTED_NPZ_COLS), f"FAIL: shape={shape}"
assert DT0 == np.dtype("<f4"), f"FAIL: dtype={DT0}"
PAY0 = data_off + 10 + hlen  # offset del primer float32 de arr_0
print(f"arr_0: shape={shape} dtype={DT0} PAY0={PAY0} ROW_BYTES={ROW_BYTES}")
print("PASS Fase 1: acceso Range/ZIP64 verificado, arr_0 STORED localizable por fila.")


In [ ]:
t0_phase2 = None
# CELDA 2 — Descarga meta.db (única descarga pesada: 3.8GB) + shas_missing (KB/MB).
# Hipótesis: sqlite exige fichero local; no hay Range posible. Con reanudación y chequeo de tamaño.
import os, urllib.request, json
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
META = os.path.join(WORK, "meta.db")
have = os.path.getsize(META) if os.path.exists(META) else 0
print("meta.db presente:", have, "/", META_SIZE)
if have != META_SIZE:
    print("descargando meta.db ...")
    req = urllib.request.Request(META_URL)
    if have > 0:
        req.add_header("Range", "bytes=%d-" % have)
    r = urllib.request.urlopen(req, timeout=120)
    print("http:", r.status)
    mode = "ab" if have > 0 and r.status == 206 else "wb"
    f = open(META, mode)
    done = os.path.getsize(META) if mode == "ab" else 0
    while True:
        b = r.read(8*2**20)
        if not b: break
        f.write(b); done += len(b)
        if (done // 2**30) != ((done - len(b)) // 2**30): print(" ...", done/2**30, "GB")
    f.close()
    assert os.path.getsize(META) == META_SIZE, "FAIL: meta.db incompleto"
print("PASS celda 2a: meta.db completo.")
MISS = os.path.join(WORK, "shas_missing_ember_features.json")
if not os.path.exists(MISS):
    urllib.request.urlretrieve(MISSING_URL, MISS)
missing = set(json.load(open(MISS)))
print("PASS celda 2b: missing set =", len(missing))


In [ ]:
SHA_ORDER = os.path.join(WORK, "train_sha_order.npy")
LAB_SURV = os.path.join(WORK, "train_lab_surv.npy")
SKIP_BUILD = SKIP_VALIDATED and os.path.exists(SHA_ORDER) and os.path.exists(LAB_SURV)
if SKIP_BUILD:
    print("artefactos existentes: se omite reconstrucción del orden train.")
else:
    # ESTRUCTURAL H6: el npz guarda vectores RAW y en orden lexicográfico de sha
    # (demostrado: fila 0 == clave LMDB mínima, fila N-1 == máxima). El rowid puro falla.
    import sqlite3, numpy as np
    con = sqlite3.connect(META)
    n_train = con.execute("SELECT COUNT(*) FROM meta WHERE rl_fs_t <= ?", (TRAIN_SPLIT,)).fetchone()[0]
    print("filas train oficial:", n_train)
    ORDER = os.path.join(WORK, "train_all_order.npy")
    mm = np.memmap(ORDER, dtype="S64", mode="w+", shape=(n_train,))
    pos = 0
    for (sha,) in con.execute("SELECT sha256 FROM meta WHERE rl_fs_t <= ? ORDER BY rowid", (TRAIN_SPLIT,)):
        mm[pos] = sha.encode(); pos += 1
        if pos % 2000000 == 0: print(" ...", pos)
    mm.flush()
    assert pos == n_train
    print("memmap train completo:", mm.shape)
    miss_arr = np.array(sorted(missing), dtype="S64")
    keep = np.ones(n_train, dtype=bool)
    CH = 1000000
    for a in range(0, n_train, CH):
        keep[a:a+CH] = ~np.isin(mm[a:a+CH], miss_arr)
    n_keep = int(keep.sum())
    print("supervivientes:", n_keep, "esperado:", EXPECTED_NPZ_ROWS)
    assert n_keep == EXPECTED_NPZ_ROWS, "FAIL H6 estructural: missing no reproduce filas"
    # supervivientes en orden rowid y reordenado lexicográficamente
    surv_unsorted = np.memmap(os.path.join(WORK, "train_sha_unsorted.npy"), dtype="S64", mode="w+", shape=(n_keep,))
    w = 0
    for a in range(0, n_train, CH):
        blk = mm[a:a+CH][keep[a:a+CH]]
        surv_unsorted[w:w+len(blk)] = blk; w += len(blk)
    surv_unsorted.flush()
    order = np.argsort(surv_unsorted)
    surv = np.memmap(SHA_ORDER, dtype="S64", mode="w+", shape=(n_keep,))
    # copiado por chunks para no duplicar 800MB en RAM
    CH2 = 2000000
    for a in range(0, n_keep, CH2):
        surv[a:a+CH2] = surv_unsorted[order[a:a+CH2]]
    surv.flush()
    print("PASS H6 estructural: orden SORTED. Artefacto:", SHA_ORDER)
    # labels en el mismo orden sorted
    labmm = np.memmap(os.path.join(WORK, "train_lab_all.npy"), dtype="i1", mode="w+", shape=(n_train,))
    pos = 0
    for (m,) in con.execute("SELECT is_malware FROM meta WHERE rl_fs_t <= ? ORDER BY rowid", (TRAIN_SPLIT,)):
        labmm[pos] = m; pos += 1
    labmm.flush()
    lab_filt = np.memmap(os.path.join(WORK, "train_lab_filt.npy"), dtype="i1", mode="w+", shape=(n_keep,))
    w = 0
    for a in range(0, n_train, CH):
        blk = labmm[a:a+CH][keep[a:a+CH]]
        lab_filt[w:w+len(blk)] = blk; w += len(blk)
    lab_filt.flush()
    labsurv = np.memmap(LAB_SURV, dtype="i1", mode="w+", shape=(n_keep,))
    for a in range(0, n_keep, CH2):
        labsurv[a:a+CH2] = lab_filt[order[a:a+CH2]]
    labsurv.flush()
    print("mapa labels sorted:", labsurv.shape, "positivos:", int(labsurv.sum()), "tasa:", float(labsurv.mean()))
    con.close()


In [ ]:
# FASE 2A — Inventario TRAIN antes de seleccionar (sin asumir 50/50)
import os, sqlite3, numpy as np, json, time, psutil
t0 = time.time()
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
SHA_ORDER = os.path.join(WORK, "train_sha_order.npy")
LAB_SURV = os.path.join(WORK, "train_lab_surv.npy")
META = os.path.join(WORK, "meta.db")
MISS = os.path.join(WORK, "shas_missing_ember_features.json")
TRAIN_SPLIT = 1543542570.0
VAL_SPLIT = 1547279640.0
EXPECTED_NPZ_ROWS = 12699013

# Verificar artefactos validados
assert os.path.exists(SHA_ORDER) and os.path.exists(LAB_SURV), "Falta train_sha_order/lab_surv — Fase 1 no completada"
assert os.path.exists(META), "Falta meta.db"
missing = set(json.load(open(MISS)))
print(f"Artefacto SHA_ORDER: {os.path.getsize(SHA_ORDER)/2**20:.1f} MB")
print(f"Artefacto LAB_SURV: {os.path.getsize(LAB_SURV)/2**20:.1f} MB")

# Carga memmap validada (SORTED)
shas = np.memmap(SHA_ORDER, dtype="S64", mode="r")
labs = np.memmap(LAB_SURV, dtype="i1", mode="r")
assert shas.shape[0]==EXPECTED_NPZ_ROWS and labs.shape[0]==EXPECTED_NPZ_ROWS
n_mal = int((labs==1).sum()); n_ben = int((labs==0).sum())
print(f"Candidatos TRAIN válidos (SORTED, sin missing): {shas.shape[0]}")
print(f"  malware: {n_mal} ({n_mal/shas.shape[0]:.2%})")
print(f"  benigno: {n_ben} ({n_ben/shas.shape[0]:.2%})")
print(f"  missing EMBER excluidos: {len(missing)} (verificado estructural)")
# timestamps: construir artefacto alineado con SORTED si no existe
TS_SORTED = os.path.join(WORK, "train_ts_sorted.npy")
if not os.path.exists(TS_SORTED):
    print("Construyendo train_ts_sorted.npy alineado con SHA_ORDER (una vez)...")
    con = sqlite3.connect(META)
    n_train = con.execute("SELECT COUNT(*) FROM meta WHERE rl_fs_t <= ?", (TRAIN_SPLIT,)).fetchone()[0]
    print(f"  filas train oficiales (con missing): {n_train}")
    # shas en orden rowid y timestamps en orden rowid
    # Usar memmap temporal en WORK (20GB disponible)
    ORDER = os.path.join(WORK, "tmp_all_shas.npy")
    TS_ALL = os.path.join(WORK, "tmp_all_ts.npy")
    mm_shas = np.memmap(ORDER, dtype="S64", mode="w+", shape=(n_train,))
    mm_ts = np.memmap(TS_ALL, dtype="f8", mode="w+", shape=(n_train,))
    pos=0
    for sha, ts in con.execute("SELECT sha256, rl_fs_t FROM meta WHERE rl_fs_t <= ? ORDER BY rowid", (TRAIN_SPLIT,)):
        mm_shas[pos]=sha.encode(); mm_ts[pos]=float(ts); pos+=1
    mm_shas.flush(); mm_ts.flush()
    assert pos==n_train
    # filtrar missing
    miss_arr=np.array(sorted(missing), dtype="S64")
    keep=np.ones(n_train, dtype=bool)
    CH=1000000
    for a in range(0,n_train,CH):
        keep[a:a+CH]=~np.isin(mm_shas[a:a+CH], miss_arr)
    n_keep=int(keep.sum())
    assert n_keep==EXPECTED_NPZ_ROWS
    # unsorted filtrados
    shas_unsorted=np.memmap(os.path.join(WORK,"tmp_filt_shas.npy"), dtype="S64", mode="w+", shape=(n_keep,))
    ts_unsorted=np.memmap(os.path.join(WORK,"tmp_filt_ts.npy"), dtype="f8", mode="w+", shape=(n_keep,))
    w=0
    for a in range(0,n_train,CH):
        blk_shas=mm_shas[a:a+CH][keep[a:a+CH]]
        blk_ts=mm_ts[a:a+CH][keep[a:a+CH]]
        shas_unsorted[w:w+len(blk_shas)]=blk_shas
        ts_unsorted[w:w+len(blk_ts)]=blk_ts
        w+=len(blk_shas)
    shas_unsorted.flush(); ts_unsorted.flush()
    # ordenar por sha para alinear con SHA_ORDER
    order=np.argsort(shas_unsorted)
    # verificar que el orden coincide con SHA_ORDER existente
    # (comparar 3 puntos + hash muestral)
    assert np.array_equal(shas_unsorted[order[:3]], shas[:3])
    assert np.array_equal(shas_unsorted[order[-3:]], shas[-3:])
    print("  orden verificado contra SHA_ORDER existente")
    # escribir timestamps ordenados
    ts_sorted=np.memmap(TS_SORTED, dtype="f8", mode="w+", shape=(n_keep,))
    CH2=2000000
    for a in range(0,n_keep,CH2):
        ts_sorted[a:a+CH2]=ts_unsorted[order[a:a+CH2]]
    ts_sorted.flush()
    # limpiar temporales
    for f in [ORDER, TS_ALL, os.path.join(WORK,"tmp_filt_shas.npy"), os.path.join(WORK,"tmp_filt_ts.npy")]:
        try: os.remove(f)
        except: pass
    con.close()
    print(f"  artefacto TS_SORTED: {os.path.getsize(TS_SORTED)/2**20:.1f} MB")
else:
    print(f"Artefacto TS_SORTED existente: {os.path.getsize(TS_SORTED)/2**20:.1f} MB")

ts = np.memmap(TS_SORTED, dtype="f8", mode="r")
print(f"rl_fs_t rango: [{ts.min():.0f}, {ts.max():.0f}]  (span {(ts.max()-ts.min())/86400:.1f} días)")
# convertir a datetime para legibilidad
import datetime
def to_date(x): return datetime.datetime.utcfromtimestamp(float(x)).strftime("%Y-%m-%d")
print(f"  fecha min: {to_date(ts.min())}  max: {to_date(ts.max())}")
print(f"  mediana: {to_date(np.median(ts))}  p10: {to_date(np.quantile(ts,0.1))}  p90: {to_date(np.quantile(ts,0.9))}")
# exclusiones
print(f"Exclusiones: missing={len(missing)}, fuera TRAIN (>={TRAIN_SPLIT:.0f}) no incluidos por query, valid/test no tocados")
# RAM
proc=psutil.Process()
print(f"RAM usada tras inventario: {proc.memory_info().rss/2**30:.2f} GB  | tiempo: {time.time()-t0:.1f}s")


In [ ]:
import time as _t
t0 = _t.time()
# FASE 2C — Muestreo estratificado 7M (seed fija, proporciones preservadas)
import numpy as np, os, time, psutil, sqlite3
SEED = 42
TARGET = 7_000_000
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
SHA_ORDER = os.path.join(WORK, "train_sha_order.npy")
LAB_SURV = os.path.join(WORK, "train_lab_surv.npy")
TS_SORTED = os.path.join(WORK, "train_ts_sorted.npy")
TRAIN_SPLIT = 1543542570.0
VAL_SPLIT = 1547279640.0

shas = np.memmap(SHA_ORDER, dtype="S64", mode="r")
labs = np.memmap(LAB_SURV, dtype="i1", mode="r")
ts = np.memmap(TS_SORTED, dtype="f8", mode="r")
N = shas.shape[0]
assert N==12699013

# Bins deciles (re-derivados determinísticamente)
qs = np.linspace(0,1,11)
bin_edges = np.quantile(ts, qs)
bins = np.digitize(ts, bin_edges[1:-1])
N_BINS=10

# estratos: (label, bin) -> 20 estratos
rng = np.random.default_rng(SEED)
# Pre-calcular índices por estrato (lista de arrays de posiciones)
stratum_indices = {}
for lab in [0,1]:
    for b in range(N_BINS):
        idx = np.where((labs==lab) & (bins==b))[0]
        stratum_indices[(lab,b)] = idx

# tamaño original por estrato
orig_counts = {k: len(v) for k,v in stratum_indices.items()}
total = N
t0=time.time()
# Asignación proporcional con corrección de redondeo
raw_alloc = {k: orig_counts[k]/total * TARGET for k in orig_counts}
alloc = {k: int(round(v)) for k,v in raw_alloc.items()}
# Ajuste para sumar exactamente TARGET
diff = TARGET - sum(alloc.values())
if diff!=0:
    # ordenar estratos por parte fraccional descendente (si sobra) o ascendente (si falta)
    frac = {k: raw_alloc[k]-alloc[k] for k in alloc}
    order = sorted(frac, key=lambda k: frac[k], reverse=(diff>0))
    for i in range(abs(diff)):
        k = order[i % len(order)]
        alloc[k] += 1 if diff>0 else -1
assert sum(alloc.values())==TARGET
# clamp a tamaño del estrato (no sobremuestrear)
for k in list(alloc):
    if alloc[k] > orig_counts[k]:
        alloc[k]=orig_counts[k]
# re-ajustar si hubo clamp
if sum(alloc.values())!=TARGET:
    # repartir el delta entre estratos con capacidad
    remaining = TARGET - sum(alloc.values())
    # capacidad = orig - alloc
    caps = {k: orig_counts[k]-alloc[k] for k in alloc}
    order = sorted(caps, key=lambda k: caps[k], reverse=True)
    for k in order:
        if remaining==0: break
        add = min(caps[k], remaining if remaining>0 else 0)
        if remaining<0:
            add = -min(alloc[k], -remaining)
        alloc[k]+=add; remaining-=add

assert sum(alloc.values())==TARGET
assert all(alloc[k]<=orig_counts[k] for k in alloc)

# Muestreo sin reemplazo por estrato
selected_lists=[]
for k, idx in stratum_indices.items():
    n = alloc[k]
    if n==0: continue
    # choice sin reemplazo, seed determinística vía rng
    sel = rng.choice(idx, size=n, replace=False)
    selected_lists.append(sel)
selected = np.concatenate(selected_lists)
# barajar globalmente para no dejar bloques por estrato (orden aleatorio pero reproducible)
rng.shuffle(selected)
assert selected.shape==(TARGET,) and len(np.unique(selected))==TARGET
# Verificar preservación de distribución
from collections import Counter
lab_sel = labs[selected]
n_mal_sel=int((lab_sel==1).sum()); n_ben_sel=int((lab_sel==0).sum())
print(f"SEED={SEED}  TARGET={TARGET}")
print(f"Original: malware {orig_counts[(1,0)]+ sum(orig_counts[(1,b)] for b in range(1,10))}?? total mal {sum(orig_counts[(1,b)] for b in range(10))} ben {sum(orig_counts[(0,b)] for b in range(10))}")
# mejor: totales
orig_mal = int((labs==1).sum()); orig_ben=int((labs==0).sum())
print(f"  Original TRAIN: malware {orig_mal} ({orig_mal/N:.2%})  benigno {orig_ben} ({orig_ben/N:.2%})")
print(f"  Seleccionado 7M: malware {n_mal_sel} ({n_mal_sel/TARGET:.2%})  benigno {n_ben_sel} ({n_ben_sel/TARGET:.2%})  delta mal {n_mal_sel/TARGET - orig_mal/N:+.2%}")
# distribución temporal resultante
bins_sel = bins[selected]
hist_sel = np.zeros((2, N_BINS), dtype=int)
for lab in [0,1]:
    hist_sel[lab]=np.bincount(bins_sel[lab_sel==lab], minlength=N_BINS)[:N_BINS]
print("\nDistribución resultante 7M por bin temporal (deciles):")
for lab in [0,1]:
    name="benigno" if lab==0 else "malware"
    print(f"  {name:8s}:", " ".join(f"{c:6d}" for c in hist_sel[lab]))

# Guardar índices
OUT_IDX = os.path.join(WORK, "train_7m_indices.npy")
np.save(OUT_IDX, selected.astype(np.uint32))
print(f"\nÍndices guardados: {OUT_IDX}  shape {selected.shape} dtype {selected.dtype}  {os.path.getsize(OUT_IDX)/2**20:.1f} MB")

# Guardar manifest Parquet
import pandas as pd
shas_sel = shas[selected].astype(str)  # S64 -> str
labs_sel = lab_sel.astype(np.int8)
ts_sel = ts[selected]
# row_idx es la posición en NPZ (que es el índice seleccionado)
row_idx = selected.astype(np.uint32)
# Verificar correspondencia row_idx <-> sha (ya ordenado)
assert np.all(shas[row_idx].astype(str)==shas_sel)

OUT_PARQ = os.path.join(WORK, "train_7m_manifest.parquet")
df = pd.DataFrame({"row_idx": row_idx, "sha256": shas_sel, "is_malware": labs_sel, "rl_fs_t": ts_sel})
# tipos eficientes
df["is_malware"]=df["is_malware"].astype("int8")
df["row_idx"]=df["row_idx"].astype("uint32")
df["rl_fs_t"]=df["rl_fs_t"].astype("float64")
df.to_parquet(OUT_PARQ, compression="zstd", index=False)
print(f"Manifest Parquet: {OUT_PARQ}  {os.path.getsize(OUT_PARQ)/2**20:.1f} MB  filas {len(df)}")
# preview CSV pequeño
preview = os.path.join(WORK, "train_7m_preview.csv")
df.head(20).to_csv(preview, index=False)
print(f"Preview CSV (20 filas): {preview}")

# Asserts fuertes
import sqlite3, json
con=sqlite3.connect(os.path.join(WORK,"meta.db"))
# 1. exactamente 7M
assert len(df)==TARGET
# 2-3. índices y shas únicos
assert df["row_idx"].nunique()==TARGET
assert df["sha256"].nunique()==TARGET
# 4. ninguna fuera TRAIN
assert df["rl_fs_t"].max() <= TRAIN_SPLIT + 1e-6
assert df["rl_fs_t"].min() >= df["rl_fs_t"].min()  # trivial, pero valida no NaN
assert df["rl_fs_t"].isna().sum()==0
# 5. ningún sha en missing
missing=set(json.load(open(os.path.join(WORK,"shas_missing_ember_features.json"))))
assert not any(s in missing for s in df["sha256"].head(1000).tolist() )  # muestral rápido
# full check por isin con memmap
miss_arr=np.array(sorted(missing), dtype="S64")
# check 7M no solape con missing via isin chunked
CH=500000
for a in range(0, TARGET, CH):
    blk=np.array(df["sha256"].iloc[a:a+CH].tolist(), dtype="S64")
    assert not np.isin(blk, miss_arr).any(), f"sha missing en bloque {a}"
# 6. labels válidos
assert set(df["is_malware"].unique()) <= {0,1}
# 7. correspondencia row_idx<->sha consistente con SHA_ORDER
shas_full=np.memmap(SHA_ORDER, dtype="S64", mode="r")
for a in range(0, min(TARGET, 10000), 2000):
    assert shas_full[df["row_idx"].iloc[a]]==df["sha256"].iloc[a].encode()
# 8. ningún solapamiento valid/test (verificar que ningún sha aparece con rl_fs_t en valid/test)
# ya garantizado por rl_fs_t <= TRAIN_SPLIT, pero verificamos contra DB que no hay shas de valid
val_shas=set(r[0] for r in con.execute("SELECT sha256 FROM meta WHERE rl_fs_t > ? LIMIT 1000", (TRAIN_SPLIT,)))
assert df["sha256"].isin(val_shas).sum()==0
print("\nAsserts fuertes: PASS")
print(f"Tiempo muestreo+guardado: {time.time()-t0:.1f}s  RAM {psutil.Process().memory_info().rss/2**30:.2f} GB")


In [ ]:
# FASE 2D — Puerta contra el dataset aprobado (numeros de Fase 2 validada).
import numpy as np, pandas as pd, os
MANIFEST = os.path.join(WORK, "train_7m_manifest.parquet")
INDICES = os.path.join(WORK, "train_7m_indices.npy")
APPROVED_MAL, APPROVED_BEN = 4187321, 2812679  # 59.82% / 40.18%, seed 42
df = pd.read_parquet(MANIFEST)
idx = np.load(INDICES)
assert len(df) == 7_000_000 and idx.shape == (7_000_000,) and idx.dtype == np.uint32
assert list(df.columns) == ["row_idx", "sha256", "is_malware", "rl_fs_t"], list(df.columns)
n_mal = int((df["is_malware"] == 1).sum())
n_ben = int((df["is_malware"] == 0).sum())
print(f"manifest: malware={n_mal} benigno={n_ben}")
assert (n_mal, n_ben) == (APPROVED_MAL, APPROVED_BEN), "FAIL: distribucion != aprobada"
assert df["row_idx"].max() < EXPECTED_NPZ_ROWS and df["row_idx"].min() >= 0
assert df["rl_fs_t"].max() <= TRAIN_SPLIT + 1e-6, "FAIL: leakage temporal"
print("PASS Fase 2: manifest 7M == dataset aprobado (seed 42, sin leakage).")


In [ ]:
# FASE 3A — Infra streaming: planificador de bloques + fetch validado + chunk size.
# A 55.1% de densidad, fusionar rangos degenera en transferencia casi total: se documenta
# y se mide honestamente (requests, bytes, MB/s). El orden de partial_fit no es relevante.
import time, psutil
import numpy as np

TARGET = 7_000_000
INDICES = os.path.join(WORK, "train_7m_indices.npy")
idx = np.load(INDICES)
assert idx.shape == (TARGET,) and idx.dtype == np.uint32
sel = np.sort(idx)  # copia ordenada solo para planificar I/O (56MB)

vm = psutil.virtual_memory()
print(f"RAM total={vm.total/2**30:.1f}GB disponible={vm.available/2**30:.1f}GB")
# Cap conservador: fetch <=256MB y buffer partial_fit <=~320MB, sin usar RAM de mas.
FETCH_SPAN_ROWS = 25000   # ~238MB por Range GET
PARTIAL_ROWS = 32768      # ~312MB por partial_fit
MAX_GAP_ROWS = 512        # fusiona huecos pequenos dentro de un span
print(f"FETCH_SPAN_ROWS={FETCH_SPAN_ROWS} (~{FETCH_SPAN_ROWS*ROW_BYTES/2**20:.0f}MB/req)")
print(f"PARTIAL_ROWS={PARTIAL_ROWS} (~{PARTIAL_ROWS*ROW_BYTES/2**20:.0f}MB/partial_fit)")

def plan_spans(s, max_gap=MAX_GAP_ROWS, max_span=FETCH_SPAN_ROWS):
    # Agrupa indices ordenados en spans [start, end) fusionando huecos <= max_gap
    # y cortando spans que excedan max_span. Devuelve lista de (start, end).
    spans = []
    start = prev = int(s[0])
    for v in s[1:]:
        v = int(v)
        if v - prev - 1 <= max_gap and v - start + 1 <= max_span:
            prev = v
            continue
        spans.append((start, prev + 1))
        start = prev = v
    spans.append((start, prev + 1))
    return spans

def fetch_span(s, e):
    # Un Range GET -> (e-s, 2381) float32 con validacion dura por chunk.
    d = npz_range(PAY0 + s * ROW_BYTES, PAY0 + e * ROW_BYTES - 1)
    X = np.frombuffer(d, dtype=np.float32).reshape(-1, N_FEATURES)
    assert X.ndim == 2 and X.shape[1] == N_FEATURES, f"FAIL chunk dim: {X.shape}"
    assert X.dtype == np.float32, f"FAIL chunk dtype: {X.dtype}"
    assert np.all(np.isfinite(X)), "FAIL chunk: NaN/Inf — abortar"
    return X

spans = plan_spans(sel)
span_rows = sum(e - s for s, e in spans)
print(f"spans={len(spans)} filas_span={span_rows} seleccionadas={TARGET} "
      f"redundancia={span_rows/TARGET:.2f}x transferencia_est={span_rows*ROW_BYTES/2**30:.1f}GB")
print("PASS Fase 3A: planificador listo.")


In [ ]:
# FASE 3B — StandardScaler streaming: spans -> mascara -> partial_fit.
from sklearn.preprocessing import StandardScaler
import psutil
scaler = StandardScaler()
io_stats = {"requests": 0, "bytes": 0, "rows_fetched": 0, "rows_selected": 0}
max_rss = 0.0
t_scaler = time.time()
buf, buf_rows, seen = [], 0, 0

def _rss():
    return psutil.Process().memory_info().rss / 2**30

for k, (s, e) in enumerate(spans):
    X = fetch_span(s, e)
    io_stats["requests"] += 1
    io_stats["bytes"] += (e - s) * ROW_BYTES
    io_stats["rows_fetched"] += (e - s)
    lo = int(np.searchsorted(sel, s))
    hi = int(np.searchsorted(sel, e))
    Xsel = X[sel[lo:hi] - s]
    io_stats["rows_selected"] += Xsel.shape[0]
    buf.append(Xsel)
    buf_rows += Xsel.shape[0]
    while buf_rows >= PARTIAL_ROWS:
        big = np.concatenate(buf, axis=0)
        use, rest = big[:PARTIAL_ROWS], big[PARTIAL_ROWS:]
        assert use.shape == (PARTIAL_ROWS, N_FEATURES) and np.all(np.isfinite(use))
        scaler.partial_fit(use)
        seen += use.shape[0]
        buf = [rest] if rest.shape[0] else []
        buf_rows = rest.shape[0]
        del big, use, rest
    max_rss = max(max_rss, _rss())
    if (k + 1) % 25 == 0 or (k + 1) == len(spans):
        el = time.time() - t_scaler
        print(f"  span {k+1}/{len(spans)} seen={seen} MB={io_stats['bytes']/2**20:.0f} "
              f"MB/s={io_stats['bytes']/2**20/max(el,1e-6):.1f} RSS={_rss():.2f}GB", flush=True)
if buf_rows:
    tail = np.concatenate(buf, axis=0)
    assert tail.shape[1] == N_FEATURES and np.all(np.isfinite(tail))
    scaler.partial_fit(tail)
    seen += tail.shape[0]
    del tail
    buf, buf_rows = [], 0
t_scaler_total = time.time() - t_scaler
max_rss = max(max_rss, _rss())
print(f"partial_fit completo: seen={seen} requests={io_stats['requests']} "
      f"GB={io_stats['bytes']/2**30:.2f} tiempo={t_scaler_total:.1f}s RSSmax={max_rss:.2f}GB")
assert io_stats["rows_selected"] == TARGET, "FAIL: filas seleccionadas != 7M"
assert scaler.n_samples_seen_ == TARGET, f"FAIL: n_samples_seen_={scaler.n_samples_seen_}"
print("PASS Fase 3B: scaler ajustado sobre 7M exactas.")


In [ ]:
# FASE 3C — Auditoria, comparativa opcional, persistencia, test 1000 y veredicto.
import hashlib, json, pickle
from datetime import datetime, timezone
import numpy as np

# --- auditoria del scaler ---
assert scaler.n_samples_seen_ == 7_000_000
mean_, var_, scale_ = scaler.mean_, scaler.var_, scaler.scale_
assert mean_.shape == (N_FEATURES,) and var_.shape == (N_FEATURES,) and scale_.shape == (N_FEATURES,)
assert np.all(np.isfinite(mean_)) and np.all(np.isfinite(var_)) and np.all(np.isfinite(scale_))
near_zero = int((var_ < 1e-12).sum())
print(f"mean_[:5]={np.array2string(mean_[:5], precision=4)} mean_[-5:]={np.array2string(mean_[-5:], precision=4)}")
print(f"var_ min={var_.min():.3g} max={var_.max():.3g} | scale_ min={scale_.min():.3g} max={scale_.max():.3g}")
print(f"features varianza~=0 (var<1e-12): {near_zero}")

# --- comparativa con scaler actual de Shadow-Net (solo si esta disponible; nunca se modifica) ---
import glob
cands = glob.glob("/kaggle/input/**/scaler*.pkl", recursive=True)
if cands:
    cur = pickle.load(open(cands[0], "rb"))
    print(f"scaler actual: {cands[0]} n_features={cur.mean_.shape[0]}")
    if cur.mean_.shape == mean_.shape:
        dmean = np.abs(cur.mean_ - mean_)
        with np.errstate(divide="ignore", invalid="ignore"):
            dscale = np.abs(cur.scale_ - scale_) / np.maximum(scale_, 1e-12)
        top = np.argsort(dmean)[-5:][::-1]
        print(f"  delta_abs mean_: mediana={np.median(dmean):.3g} max={dmean.max():.3g} top5_idx={top.tolist()}")
        print(f"  delta_rel scale_: mediana={np.median(dscale):.3g} max={dscale.max():.3g}")
    else:
        print("  dims distintas: comparativa omitida (no bloqueante)")
else:
    print("scaler actual no disponible en Kaggle: comparativa omitida (no bloqueante)")

# --- persistencia ---
PKL = os.path.join(WORK, "scaler_sorel_7m_v1.1.pkl")
JSN = os.path.join(WORK, "scaler_sorel_7m_v1.1.json")
with open(PKL, "wb") as f:
    pickle.dump(scaler, f, protocol=4)
sha_pkl = hashlib.sha256(open(PKL, "rb").read()).hexdigest()
meta = {
    "version": "scaler_sorel_7m_v1.1",
    "dataset": "SOREL-20M train split oficial, seleccion 7M seed 42 (59.82% malware)",
    "n_samples": 7_000_000,
    "n_features": N_FEATURES,
    "seed": 42,
    "method": "StandardScaler.partial_fit streaming por spans Range (STORED float32)",
    "chunk_size": {"fetch_span_rows": FETCH_SPAN_ROWS, "partial_rows": PARTIAL_ROWS, "max_gap_rows": MAX_GAP_ROWS},
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "sha256_pkl": sha_pkl,
    "versiones": {"numpy": np.__version__, "sklearn": __import__("sklearn").__version__, "python": sys.version.split()[0]},
    "io": {**io_stats, "gb_transferidos": io_stats["bytes"] / 2**30},
    "tiempo_total_s": t_scaler_total,
    "ram_max_gb": max_rss,
    "auditoria": {"near_zero_var": near_zero, "scale_min": float(scale_.min()), "scale_max": float(scale_.max())},
}
open(JSN, "w").write(json.dumps(meta, indent=2))
assert os.path.getsize(PKL) > 0 and os.path.getsize(JSN) > 0
print(f"artefactos: {PKL} ({os.path.getsize(PKL)/1024:.1f}KB, sha256={sha_pkl[:16]}...) {JSN}")
# recarga y verificacion post-guardado
sc2 = pickle.load(open(PKL, "rb"))
assert sc2.n_samples_seen_ == 7_000_000 and np.array_equal(sc2.mean_, mean_)
print("recarga post-guardado: OK")

# --- test 1000 muestras (sanity, no 0/1 exactos) ---
rng_t = np.random.default_rng(12345)
tpos = np.sort(rng_t.choice(TARGET, 1000, replace=False))
trows = np.sort(idx[tpos])
tspans = plan_spans(trows, max_gap=2048, max_span=50000)
parts = []
for s, e in tspans:
    X = fetch_span(s, e)
    lo = int(np.searchsorted(trows, s)); hi = int(np.searchsorted(trows, e))
    parts.append(X[trows[lo:hi] - s])
Xt = np.concatenate(parts, axis=0)
assert Xt.shape == (1000, N_FEATURES)
mean_before = scaler.mean_.copy()
seen_before = scaler.n_samples_seen_
Zt = scaler.transform(Xt)
assert Zt.shape == (1000, N_FEATURES) and np.all(np.isfinite(Zt))
assert scaler.n_samples_seen_ == seen_before and np.array_equal(scaler.mean_, mean_before), "FAIL: scaler mutado"
print(f"test 1000: shape={Zt.shape} finite=True scaler_no_mutado=True (sanity, sin exigir media 0/1)")
print("STANDARD SCALER 2381: PASS")


## Fase 4 — Overlay feature engineering (cerrada)

En esta fase construi el contrato `ShadowNetFeatures_v1.1 = EMBER_2381 + OVERLAY_6`. Uso solo el
bloque General de EMBER (indices 616:626), cuya semantica es identica en SOREL y en produccion;
excluyo los indices de seccion (688:943) porque su layout no esta verificado como identico entre
ambos. El overlay son 6 features baratas e interpretables.


# FASE 4 — Bloque OVERLAY_6: contrato ShadowNetFeatures_v1.1 = EMBER_2381 + OVERLAY_6.
# Dejo documentada una limitacion que no pretendo ocultar: el OverlayAnalyzer forense opera
# sobre BYTES crudos (entropia del overlay, strings, firmas MZ, PEs embebidos) y esos campos
# NO son reproducibles desde el vector EMBER-2381, asi que no los incluyo. Uso solo el bloque
# General (indices 616:626), de semantica EMBER identica en SOREL y en produccion. N=6.


In [ ]:
# FASE 4B — Contrato overlay: nombres, orden y funcion pura (identica a core/overlay_features.py).
import numpy as np

IDX_SIZE, IDX_VSIZE, IDX_IMPORTS, IDX_CERT = 616, 617, 620, 623
OVERLAY_FEATURE_NAMES = [
    "slack_ratio",
    "slack_bytes_log",
    "file_size_log",
    "imports_log",
    "has_cert",
    "stub_overlay_pattern",
]
N_OVERLAY = len(OVERLAY_FEATURE_NAMES)
assert N_OVERLAY == 6 and N_OVERLAY <= 16

def compute_overlay_block(x):
    v = np.asarray(x, dtype=np.float64)
    single = v.ndim == 1
    if single:
        v = v[np.newaxis, :]
    assert v.shape[1] >= 624, f"vector corto: {v.shape[1]}"
    size = np.maximum(v[:, IDX_SIZE], 0.0)
    vsize = np.maximum(v[:, IDX_VSIZE], 0.0)
    imports = np.maximum(v[:, IDX_IMPORTS], 0.0)
    cert = v[:, IDX_CERT]
    slack = np.where(size > 0, np.maximum(size - vsize, 0.0), 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        slack_ratio = np.where(size > 0, slack / np.maximum(size, 1.0), 0.0)
    slack_ratio = np.clip(slack_ratio, 0.0, 1.0)
    out = np.empty((v.shape[0], N_OVERLAY), dtype=np.float64)
    out[:, 0] = slack_ratio
    out[:, 1] = np.log1p(slack)
    out[:, 2] = np.log1p(size)
    out[:, 3] = np.log1p(imports)
    out[:, 4] = (cert != 0.0).astype(np.float64)
    out[:, 5] = ((imports <= 5) & (slack_ratio > 0.5)).astype(np.float64)
    res = out.astype(np.float32)
    return res[0] if single else res

# Casos deterministas conocidos (sinteticos, sin descargas).
def _v(*, size=0.0, vsize=0.0, imports=0.0, cert=0.0):
    v = np.zeros(2381, dtype=np.float32)
    v[616], v[617], v[620], v[623] = size, vsize, imports, cert
    return v

o = compute_overlay_block(_v(size=20.9*1024*1024, vsize=0.3*1024*1024, imports=3, cert=0))
assert o.shape == (6,) and 0.98 < o[0] < 0.99 and o[1] > 16.0 and o[4] == 0.0 and o[5] == 1.0, o
o = compute_overlay_block(_v(size=1024*1024, vsize=1024*1024, imports=50, cert=1))
assert o[0] == 0.0 and o[1] == 0.0 and o[4] == 1.0 and o[5] == 0.0, o
assert np.all(compute_overlay_block(np.zeros(2381, dtype=np.float32)) == 0.0)
o = compute_overlay_block(_v(size=1000.0, vsize=5000.0, imports=10, cert=0))
assert o[0] == 0.0 and o[5] == 0.0, o
print("PASS Fase 4B: contrato N=6, orden fijo, 4 casos deterministas OK.")

In [ ]:
# FASE 4C — Muestra representativa 50k, validacion, scaler overlay propio y artefactos.
# Muestreo: 50 bloques contiguos de 1000 filas espaciados uniformemente. Valido porque
# las filas NPZ estan en orden hash-sha, independiente de label y tiempo (50 requests).
import hashlib, json, pickle, time
from datetime import datetime, timezone
from sklearn.preprocessing import StandardScaler
import numpy as np

LAB_SURV_P = os.path.join(WORK, "train_lab_surv.npy")
TS_SORTED_P = os.path.join(WORK, "train_ts_sorted.npy")
for p in (MANIFEST, INDICES, LAB_SURV_P, TS_SORTED_P):
    assert os.path.exists(p), f"falta artefacto Fase 2: {p}"

NBLOCKS, BLOCK = 50, 1000
starts = np.linspace(0, EXPECTED_NPZ_ROWS - BLOCK, NBLOCKS).astype(int)
t4 = time.time()
io4 = {"requests": 0, "bytes": 0}
chunks = []
for s in starts:
    s = int(s)
    X = fetch_span(s, s + BLOCK)
    io4["requests"] += 1
    io4["bytes"] += BLOCK * ROW_BYTES
    chunks.append(X)
Xb = np.concatenate(chunks, axis=0)
del chunks
assert Xb.shape == (NBLOCKS * BLOCK, N_FEATURES), Xb.shape
print(f"muestra: {Xb.shape} ({Xb.nbytes/2**20:.0f}MB, {io4['requests']} requests)")

O = compute_overlay_block(Xb)
del Xb
assert O.shape == (50000, N_OVERLAY) and O.dtype == np.float32, (O.shape, O.dtype)
assert np.all(np.isfinite(O)), "FAIL: NaN/Inf en overlay"
assert O[:, 0].min() >= 0.0 and O[:, 0].max() <= 1.0, "FAIL: slack_ratio fuera de [0,1]"
assert O[:, 1:4].min() >= 0.0, "FAIL: logs negativos"
assert set(np.unique(O[:, 4])).issubset({0.0, 1.0}), "FAIL: has_cert no binario"
assert set(np.unique(O[:, 5])).issubset({0.0, 1.0}), "FAIL: stub no binario"
present = [(O[:, j] != 0).mean() for j in range(N_OVERLAY)]
print("presentes por feature:", [f"{n}={p:.1%}" for n, p in zip(OVERLAY_FEATURE_NAMES, present)])
for j, n in enumerate(OVERLAY_FEATURE_NAMES):
    assert np.unique(O[:, j]).size > 1, f"FAIL: feature constante inesperada: {n}"

labmm = np.memmap(LAB_SURV_P, dtype="i1", mode="r")
tsmm = np.memmap(TS_SORTED_P, dtype="f8", mode="r")
rowpos = np.concatenate([np.arange(int(s), int(s) + BLOCK) for s in starts])
y = labmm[rowpos].astype(int)
mal_rate = float(y.mean())
print(f"malware en muestra: {mal_rate:.2%} (ref TRAIN 59.82%)")
assert abs(mal_rate - 0.5982) < 0.02 and y.sum() > 0 and (y == 0).sum() > 0
tspan = float(tsmm[rowpos].max() - tsmm[rowpos].min())
print(f"cobertura temporal: {tspan/86400:.0f} dias (ref 698)")
assert tspan > 500 * 86400, "FAIL: muestra temporalmente sesgada"
for lab, tag in ((1, "malware"), (0, "benigno")):
    m = O[y == lab].mean(axis=0)
    print(f"  media {tag:8s}: " + " ".join(f"{v:.3g}" for v in m))

scaler_ov = StandardScaler()
scaler_ov.fit(O)
assert scaler_ov.n_features_in_ == N_OVERLAY
assert np.all(np.isfinite(scaler_ov.mean_)) and np.all(np.isfinite(scaler_ov.scale_))
print(f"overlay scaler: mean={np.array2string(scaler_ov.mean_, precision=3)} scale={np.array2string(scaler_ov.scale_, precision=3)}")

PKL_OV = os.path.join(WORK, "scaler_overlay_sorel_7m_v1.1.pkl")
JSN_OV = os.path.join(WORK, "scaler_overlay_sorel_7m_v1.1.json")
with open(PKL_OV, "wb") as f:
    pickle.dump(scaler_ov, f, protocol=4)
sha_ov = hashlib.sha256(open(PKL_OV, "rb").read()).hexdigest()
meta_ov = {
    "contract": "ShadowNetFeatures_v1.1",
    "features": "EMBER_2381 (scaler_sorel_7m_v1.1, intacto) + OVERLAY_6",
    "feature_names": OVERLAY_FEATURE_NAMES,
    "order": list(range(N_OVERLAY)),
    "N": N_OVERLAY,
    "ember_indices": [IDX_SIZE, IDX_VSIZE, IDX_IMPORTS, IDX_CERT],
    "normalization": "StandardScaler propio (fit en muestra 50k determinista, sin tocar scaler 2381)",
    "sample": {"n": 50000, "blocks": NBLOCKS, "block_rows": BLOCK, "method": "linspace-uniforme (sin RNG)"},
    "seed": "determinista-sin-RNG (linspace); manifest base seed 42",
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "sha256_pkl": sha_ov,
    "versiones": {"numpy": np.__version__, "sklearn": __import__("sklearn").__version__, "python": sys.version.split()[0]},
    "io": io4,
    "tiempo_s": round(time.time() - t4, 1),
}
open(JSN_OV, "w").write(json.dumps(meta_ov, indent=2))
assert os.path.getsize(PKL_OV) > 0 and os.path.getsize(JSN_OV) > 0
print(f"artefactos: {PKL_OV} sha256={sha_ov[:16]}... + metadata JSON")
print("PASS Fase 4C: validacion minima OK, scaler overlay guardado.")

In [ ]:
# FASE 4D — Puerta final del contrato.
import hashlib, json, pickle
import numpy as np

PKL_OV = os.path.join(WORK, "scaler_overlay_sorel_7m_v1.1.pkl")
JSN_OV = os.path.join(WORK, "scaler_overlay_sorel_7m_v1.1.json")
sc = pickle.load(open(PKL_OV, "rb"))
assert sc.n_features_in_ == 6, sc.n_features_in_
meta_ov = json.load(open(JSN_OV, "r"))
assert meta_ov["feature_names"] == OVERLAY_FEATURE_NAMES, "FAIL: orden/nombres != contrato"
assert meta_ov["N"] == 6 and meta_ov["contract"] == "ShadowNetFeatures_v1.1"
assert meta_ov["sha256_pkl"] == hashlib.sha256(open(PKL_OV, "rb").read()).hexdigest()
# La 2381 sigue intacta: el scaler v1.1 no fue tocado en esta fase.
print("ShadowNetFeatures_v1.1 = EMBER_2381 + OVERLAY_6")
print("orden:", OVERLAY_FEATURE_NAMES)
print("OVERLAY FEATURE CONTRACT: PASS")

## Fase 5 — Entrenamiento MLP 7M sobre ShadowNetFeatures_v1.1 (2387)

En esta fase entreno el MLP sobre el vector de 2387 dimensiones, reutilizando sin modificar los
artefactos que ya cerre en las fases anteriores:

- **5A** — detecto el hardware (P100 / T4 / T4x2 / CPU), configuro multi-GPU si hay dos T4,
  cargo los scalers en solo lectura, construyo el split temporal 90/10 y ejecuto un sanity real
  (forward + loss + backward + optimizer.step) antes de gastar horas de entrenamiento.
- **5B** — construyo una cache en disco con una sola pasada HTTP sobre los spans y la reutilizo
  para las 2 epocas completas y para la validacion. Asi evito re-descargar ~112 GB por epoca,
  que fue el cuello real medido en la ejecucion anterior. Cada epoca recorre completo su conjunto.
- **5C** — evaluo el checkpoint `final` y el `best`, calculo metricas completas con threshold 0.5
  (sin tuning), guardo `metrics.json` y `metadata.json` con hashes, y verifico recargando el `.pth`.
- **5D** — genero las 10 graficas del experimento como evidencia reproducible.

No comparo contra modelos ni fases anteriores: todas las metricas y graficas describen unicamente
este entrenamiento. No toco el modelo de produccion de 2381 features.


In [ ]:
# FASE 5A (v5) — Setup: hardware + multi-GPU, scalers (solo lectura), split temporal,
# modelo ShadowNetMLP, sanity real (forward + loss + backward + optimizer.step) y planificacion
# de la cache. Si el sanity falla, detengo la ejecucion aqui.
import time, hashlib, json, pickle, os, shutil
import numpy as np, pandas as pd
import torch, torch.nn as nn
T5A = time.time()
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

# --- hardware: detecto primero; AMP solo donde es estable (P100 sm_60 sin AMP) ---
USE_CUDA = torch.cuda.is_available()
N_GPU = torch.cuda.device_count() if USE_CUDA else 0
DEV = torch.device("cuda" if USE_CUDA else "cpu")
print(f"torch={torch.__version__} cuda_build={torch.version.cuda} disponible={USE_CUDA} n_gpu={N_GPU}")
GPU_DESC = []
if USE_CUDA:
    for i in range(N_GPU):
        pr = torch.cuda.get_device_properties(i)
        cp = torch.cuda.get_device_capability(i)
        GPU_DESC.append({"index": i, "name": pr.name,
                         "capability": f"{cp[0]}.{cp[1]}",
                         "vram_gb": round(pr.total_memory / 2**30, 1)})
        print(f"  gpu{i}: {pr.name} cap={cp} vram={pr.total_memory/2**30:.1f}GB")
    try:
        # La reduccion (.sum / .ne) es justo la que no tenia kernel en cu128 sobre sm_60.
        _a = torch.ones(256, 256, device="cuda")
        _s = float((_a @ _a).sum().cpu())
        _n = float(_a.ne(0).float().sum().cpu())
        assert abs(_s - 256**3) < 1.0 and _n == 256 * 256
        print("smoke CUDA OK: matmul + reduccion utilizables en la GPU asignada.")
    except Exception as e:
        print(f"AVISO: CUDA no utilizable ({type(e).__name__}: {str(e)[:120]}), fallback a CPU.")
        USE_CUDA, N_GPU, DEV = False, 0, torch.device("cpu")
if not USE_CUDA:
    torch.set_num_threads(os.cpu_count() or 4)
    print(f"CPU: threads={torch.get_num_threads()} (documentado, mas lento que GPU).")
CAP = torch.cuda.get_device_capability(0) if USE_CUDA else (0, 0)
# P100 (sm_60) no acelera fp16 de forma fiable: sin AMP. T4 (sm_75) si: AMP estable.
USE_AMP = bool(USE_CUDA and CAP[0] >= 7)
print(f"device={DEV} capability={CAP} AMP={USE_AMP}")

# --- scalers: solo lectura, jamas re-fit (Fases 3-4 cerradas) ---
EXP_EMBER = "a20feb5b227f2ece4eb044e67384449820986abc0d3235baef81cb1b482f5dd4"
EXP_OVER = "42ab1cfd3e745f95f19b9c2ba94282af9bd6dbce573506ed954928439778e706"
def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
P_EMB = os.path.join(WORK, "scaler_sorel_7m_v1.1.pkl")
P_OVR = os.path.join(WORK, "scaler_overlay_sorel_7m_v1.1.pkl")
for p, exp in ((P_EMB, EXP_EMBER), (P_OVR, EXP_OVER)):
    assert os.path.exists(p), f"FAIL: falta {p}"
    assert _sha(p) == exp, f"FAIL: sha distinto en {p}"
sc_emb = pickle.load(open(P_EMB, "rb"))
sc_ovr = pickle.load(open(P_OVR, "rb"))
assert sc_emb.n_features_in_ == 2381 and sc_ovr.n_features_in_ == 6
MEAN_EMB, SCALE_EMB = sc_emb.mean_.copy(), sc_emb.scale_.copy()
MEAN_OVR, SCALE_OVR = sc_ovr.mean_.copy(), sc_ovr.scale_.copy()
print("scalers OK: EMBER(2381) + OVERLAY(6), hashes verificados, solo lectura.")

# --- split temporal reproducible desde el manifest (sin RNG, sin leakage) ---
MANIFEST = os.path.join(WORK, "train_7m_manifest.parquet")
INDICES = os.path.join(WORK, "train_7m_indices.npy")
df = pd.read_parquet(MANIFEST, columns=["row_idx", "is_malware", "rl_fs_t"])
assert len(df) == 7_000_000
assert int((df["is_malware"] == 1).sum()) == 4187321
assert int((df["is_malware"] == 0).sum()) == 2812679
ts = df["rl_fs_t"].to_numpy()
order = np.argsort(ts, kind="stable")
N_TRAIN = 6_300_000
rows = df["row_idx"].to_numpy(dtype=np.uint32)
labs_all = np.memmap(LAB_SURV, dtype="i1", mode="r")
train_rows = rows[order[:N_TRAIN]]
val_rows = rows[order[N_TRAIN:]]
split_of = np.full(EXPECTED_NPZ_ROWS, -1, dtype=np.int8)
split_of[train_rows] = 0
split_of[val_rows] = 1
assert (split_of == 0).sum() == N_TRAIN and (split_of == 1).sum() == 700_000
assert set(np.load(INDICES).tolist()) == set(train_rows.tolist()) | set(val_rows.tolist())
t_bound = ts[order[N_TRAIN - 1]]
t_train_min, t_train_max = ts[order[0]], ts[order[N_TRAIN - 1]]
t_val_min, t_val_max = ts[order[N_TRAIN]], ts[order[-1]]
print(f"split temporal: train={N_TRAIN} (mal={int(labs_all[train_rows].sum())}) "
      f"val=700000 (mal={int(labs_all[val_rows].sum())}) corte_ts={t_bound:.0f} (val>={t_bound:.0f})")
print(f"  train ts=[{t_train_min:.0f},{t_train_max:.0f}]  val ts=[{t_val_min:.0f},{t_val_max:.0f}]")
print("limite documentado: val esta dentro del rango TRAIN; test futuro real = SOREL test.")
del df, ts, order, rows

# --- modelo: base limpio (ONNX-ready) + wrapper multi-GPU si Kaggle asigna 2 T4 ---
class ShadowNetMLP(nn.Module):
    def __init__(self, d_in=2387):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 1),  # logit -> BCEWithLogitsLoss
        )
    def forward(self, x):
        return self.net(x)  # [batch, 1] logits (contrato ONNX)
base_model = ShadowNetMLP(2387).to(DEV)
model = nn.DataParallel(base_model) if N_GPU > 1 else base_model
if N_GPU > 1:
    print(f"multi-GPU: nn.DataParallel sobre {N_GPU} GPUs (guardo el state_dict del modelo base).")
N_PARAMS = sum(p.numel() for p in base_model.parameters())
assert N_PARAMS == 1388801, f"FAIL arquitectura: {N_PARAMS}"
print(f"parametros: {N_PARAMS} (~{N_PARAMS/1e6:.2f}M)")
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(base_model.parameters(), lr=1e-3)
scaler_amp = torch.amp.GradScaler("cuda", enabled=USE_AMP) if USE_AMP else None
print("optimizer=Adam lr=1e-3 scheduler=ninguno epochs=2 batch=8192")

# --- pipeline: EMBER->scaler EMBER, OVERLAY->scaler OVERLAY, concat 2387 ---
OVL_COLS = (616, 617, 620, 623)
def build_batch(Xraw, prows):
    # Xraw: (n,2381) float32 crudo del NPZ; prows: filas NPZ. Devuelve tensores en DEV.
    Xe = sc_emb.transform(Xraw)
    size, vsize, nimp, cert = (Xraw[:, c] for c in OVL_COLS)
    slack = np.maximum(size - vsize, 0.0)
    ratio = np.where(size > 0, slack / np.maximum(size, 1e-6), 0.0)
    stub = ((nimp <= 5) & (ratio > 0.5)).astype(np.float32)
    Oraw = np.stack([ratio, np.log1p(slack), np.log1p(np.maximum(size, 0.0)),
                     np.log1p(np.maximum(nimp, 0.0)), (cert > 0).astype(np.float32), stub],
                    axis=1).astype(np.float32)
    Xo = sc_ovr.transform(Oraw)
    X = np.concatenate([Xe, Xo], axis=1).astype(np.float32)
    y = labs_all[prows].astype(np.float32)
    return torch.from_numpy(X).to(DEV), torch.from_numpy(y).to(DEV)

# --- sanity real obligatorio: forward + loss + backward + optimizer.step ---
sel = np.sort(np.load(INDICES))
spans = plan_spans(sel)
s0, e0 = spans[0]
n_s = min(8192, e0 - s0)
Xr = fetch_span(s0, s0 + n_s)
pr = np.arange(s0, s0 + n_s, dtype=np.uint32)
Xb, yb = build_batch(Xr, pr)
assert Xb.shape == (n_s, 2387) and Xb.dtype == torch.float32, (Xb.shape, Xb.dtype)
assert yb.shape == (n_s,) and set(yb.unique().tolist()) <= {0.0, 1.0}
assert torch.all(torch.isfinite(Xb)), "FAIL sanity: no finito"
assert Xb.shape[1] - 6 == 2381
assert np.array_equal(sc_emb.mean_, MEAN_EMB) and np.array_equal(sc_emb.scale_, SCALE_EMB)
assert np.array_equal(sc_ovr.mean_, MEAN_OVR) and np.array_equal(sc_ovr.scale_, SCALE_OVR)
_init_state = {k: v.detach().clone() for k, v in base_model.state_dict().items()}
base_model.train()
optimizer.zero_grad(set_to_none=True)
_logits = model(Xb)
assert _logits.shape == (n_s, 1), f"FAIL sanity: output {tuple(_logits.shape)} != (n,1)"
_loss = criterion(_logits, yb.unsqueeze(1))
_loss.backward()
optimizer.step()
assert bool(torch.isfinite(_loss)), "FAIL sanity: loss no finita"
base_model.load_state_dict(_init_state)  # restauro: el sanity no altera los pesos
print(f"GPU sanity check: PASS | X{Xb.shape} y{yb.shape} out{tuple(_logits.shape)} "
      f"mal={int(yb.sum())} loss={float(_loss):.4f} backward+optimizer.step OK")
del Xr, Xb, yb, _logits, _loss, _init_state

# --- planificacion de cache: una sola descarga alimenta 2 epocas + validacion ---
CACHE_BYTES = 7_000_000 * 2387 * 4
def _pick_cache_dir():
    for d in ("/tmp", "/kaggle/working"):
        try:
            free = shutil.disk_usage(d).free
            if free > CACHE_BYTES + 25 * 2**30:
                p = os.path.join(d, "shadownet_v5_cache")
                os.makedirs(p, exist_ok=True)
                return p, free
        except Exception:
            pass
    return None, 0
CACHE_DIR, CACHE_FREE = _pick_cache_dir()
USE_CACHE = CACHE_DIR is not None
X_CACHE = Y_CACHE = None
split_slot = None
if USE_CACHE:
    X_CACHE = np.memmap(os.path.join(CACHE_DIR, "X2387.f32"), dtype=np.float32, mode="w+",
                        shape=(7_000_000, 2387))
    Y_CACHE = np.memmap(os.path.join(CACHE_DIR, "y.i1"), dtype=np.int8, mode="w+",
                        shape=(7_000_000,))
    # El slot i de la cache es la i-esima fila seleccionada en orden NPZ ascendente
    # (= np.sort(INDICES)): asi la construccion y la lectura son secuenciales (sin I/O aleatorio).
    split_slot = split_of[sel].astype(np.int8)
    assert int((split_slot == 0).sum()) == 6_300_000
    assert int((split_slot == 1).sum()) == 700_000
    print(f"cache_dir={CACHE_DIR} libre={CACHE_FREE/2**30:.0f}GB necesita={CACHE_BYTES/2**30:.0f}GB")
else:
    print("AVISO: sin espacio para cache; usare streaming por spans (mas lento, documentado).")
print(f"PASS Fase 5A: setup listo ({time.time()-T5A:.0f}s).")


In [ ]:
# FASE 5B (v5) — Cache unico (1 pasada HTTP) + 2 epocas completas.
# Construyo la cache 7M x 2387 con una sola descarga de spans y la reutilizo para las 2 epocas
# y para la validacion. Cada epoca recorre completo su conjunto de entrenamiento. Registro
# metricas completas por epoca con threshold fijo 0.5.
import time, csv
import numpy as np, torch
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
T5B = time.time()
EPOCHS, BATCH = 2, 8192
CKPT_BEST = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_v5_best.pth")
CKPT_FINAL = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_v5_final.pth")
HIST = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_v5_history.csv")
THRESHOLD = 0.5
t_fetch = t_proc = t_write = t_train = t_val = 0.0
n_fetch_bytes = 0

def _metrics_from_scores(y, p, thr=THRESHOLD):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()
    return {"accuracy": float(accuracy_score(y, pred)),
            "precision": float(precision_score(y, pred, zero_division=0)),
            "recall": float(recall_score(y, pred, zero_division=0)),
            "f1": float(f1_score(y, pred, zero_division=0)),
            "roc_auc": float(roc_auc_score(y, p)),
            "pr_auc": float(average_precision_score(y, p)),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def _metrics_from_counts(n, ttn, tfp, tfn, ttp):
    acc = (ttp + ttn) / n
    prec = ttp / (ttp + tfp) if (ttp + tfp) else 0.0
    rec = ttp / (ttp + tfn) if (ttp + tfn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {"accuracy": float(acc), "precision": float(prec), "recall": float(rec),
            "f1": float(f1), "tn": int(ttn), "fp": int(tfp), "fn": int(tfn), "tp": int(ttp)}

def _optim_step(xb, yb):
    # Un paso de optimizacion; devuelve (loss, n, (tn,fp,fn,tp)) para metricas de train.
    global t_train
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)
    if USE_AMP:
        with torch.amp.autocast("cuda", enabled=True):
            logits = model(xb)
            loss = criterion(logits, yb.unsqueeze(1))
        scaler_amp.scale(loss).backward()
        scaler_amp.step(optimizer)
        scaler_amp.update()
    else:
        logits = model(xb)
        loss = criterion(logits, yb.unsqueeze(1))
        loss.backward()
        optimizer.step()
    t_train += time.time() - t0
    with torch.no_grad():
        p = logits.reshape(-1) > 0
        y = yb > 0.5
        c = (int((~p & ~y).sum().cpu()), int((p & ~y).sum().cpu()),
             int((~p & y).sum().cpu()), int((p & y).sum().cpu()))
    return float(loss.detach().cpu()), len(xb), c

hist_rows = []
best_val = float("inf")
val_probs_last = None
val_true_last = None

if USE_CACHE:
    # ---------- Paso 1: construyo la cache con una sola pasada HTTP ----------
    written = 0
    for k, (s, e) in enumerate(spans):
        t0 = time.time()
        Xr = fetch_span(s, e)
        t_fetch += time.time() - t0
        n_fetch_bytes += Xr.nbytes
        prows = np.arange(s, e, dtype=np.uint32)
        m = split_of[prows]
        selm = m != -1
        if selm.any():
            t0 = time.time()
            Xb, yb = build_batch(Xr[selm], prows[selm])
            t_proc += time.time() - t0
            n_new = int(selm.sum())
            t0 = time.time()
            X_CACHE[written:written + n_new] = Xb.cpu().numpy()
            Y_CACHE[written:written + n_new] = yb.cpu().numpy().astype(np.int8)
            t_write += time.time() - t0
            written += n_new
        del Xr
        if (k + 1) % 50 == 0 or (k + 1) == len(spans):
            print(f"  cache {k+1}/{len(spans)} slots={written} ({time.time()-T5B:.0f}s)", flush=True)
    X_CACHE.flush()
    Y_CACHE.flush()
    assert written == 7_000_000, f"FAIL cache: slots={written}"
    assert int((np.asarray(Y_CACHE[:]) >= 0).all()) == 1, "FAIL cache: labels invalidas"
    print(f"cache completa: fetch={t_fetch:.0f}s proc={t_proc:.0f}s write={t_write:.0f}s")
    train_slots = np.where(split_slot == 0)[0]
    val_slots = np.where(split_slot == 1)[0]

    def train_epoch():
        model.train()
        run_loss, n, steps = 0.0, 0, 0
        ttn = tfp = tfn = ttp = 0
        for a in range(0, len(train_slots), BATCH):
            sl = train_slots[a:a + BATCH]
            xb = torch.from_numpy(np.array(X_CACHE[sl])).to(DEV)
            yb = torch.from_numpy(np.array(Y_CACHE[sl]).astype(np.float32)).to(DEV)
            l, nb, (a0, a1, a2, a3) = _optim_step(xb, yb)
            run_loss += l * nb
            n += nb
            steps += 1
            ttn += a0
            tfp += a1
            tfn += a2
            ttp += a3
            del xb, yb
        return run_loss / n, n, steps, _metrics_from_counts(n, ttn, tfp, tfn, ttp)

    def val_epoch():
        global t_val
        model.eval()
        probs = np.empty(len(val_slots), dtype=np.float32)
        w = 0
        vloss = 0.0
        with torch.no_grad():
            for a in range(0, len(val_slots), BATCH):
                t0 = time.time()
                sl = val_slots[a:a + BATCH]
                xb = torch.from_numpy(np.array(X_CACHE[sl])).to(DEV)
                yb = torch.from_numpy(np.array(Y_CACHE[sl]).astype(np.float32)).to(DEV)
                logits = model(xb)
                vloss += float(criterion(logits, yb.unsqueeze(1)).cpu()) * len(xb)
                probs[w:w + len(xb)] = torch.sigmoid(logits).reshape(-1).float().cpu().numpy()
                w += len(xb)
                t_val += time.time() - t0
                del xb, yb, logits
        y = np.array(Y_CACHE[val_slots]).astype(int)
        return probs, y, vloss / w

else:
    # ---------- Ruta B: streaming (solo si no hubo espacio para cache) ----------
    def train_epoch():
        global t_fetch, t_proc
        model.train()
        run_loss, n, steps = 0.0, 0, 0
        ttn = tfp = tfn = ttp = 0
        buf_x, buf_y, bn = [], [], 0
        for (s, e) in spans:
            t0 = time.time()
            Xr = fetch_span(s, e)
            t_fetch += time.time() - t0
            prows = np.arange(s, e, dtype=np.uint32)
            idx = np.where(split_of[prows] == 0)[0]
            if len(idx):
                t0 = time.time()
                Xb, yb = build_batch(Xr[idx], prows[idx])
                t_proc += time.time() - t0
                buf_x.append(Xb.cpu())
                buf_y.append(yb.cpu())
                bn += len(Xb)
            del Xr
            while bn >= BATCH:
                Xc = torch.cat(buf_x, 0)
                yc = torch.cat(buf_y, 0)
                xb, yb2 = Xc[:BATCH].to(DEV), yc[:BATCH].to(DEV)
                rx, ry = Xc[BATCH:], yc[BATCH:]
                buf_x, buf_y = ([rx] if len(rx) else []), ([ry] if len(ry) else [])
                bn = len(rx)
                l, nb, (a0, a1, a2, a3) = _optim_step(xb, yb2)
                run_loss += l * nb
                n += nb
                steps += 1
                ttn += a0
                tfp += a1
                tfn += a2
                ttp += a3
                del xb, yb2, Xc, yc
        if bn:
            Xc = torch.cat(buf_x, 0).to(DEV)
            yc = torch.cat(buf_y, 0).to(DEV)
            l, nb, (a0, a1, a2, a3) = _optim_step(Xc, yc)
            run_loss += l * nb
            n += nb
            steps += 1
            ttn += a0
            tfp += a1
            tfn += a2
            ttp += a3
            del Xc, yc, buf_x, buf_y
        return run_loss / n, n, steps, _metrics_from_counts(n, ttn, tfp, tfn, ttp)

    def val_epoch():
        global t_fetch, t_proc, t_val
        model.eval()
        probs, ys, vloss, vn = [], [], 0.0, 0
        with torch.no_grad():
            for (s, e) in spans:
                t0 = time.time()
                Xr = fetch_span(s, e)
                t_fetch += time.time() - t0
                prows = np.arange(s, e, dtype=np.uint32)
                idx = np.where(split_of[prows] == 1)[0]
                if len(idx):
                    Xb, yb = build_batch(Xr[idx], prows[idx])
                    for a in range(0, len(Xb), BATCH):
                        logits = model(Xb[a:a + BATCH])
                        vloss += float(criterion(logits, yb[a:a + BATCH].unsqueeze(1)).cpu()) * len(logits)
                        probs.append(torch.sigmoid(logits).reshape(-1).float().cpu().numpy())
                        ys.append(yb[a:a + BATCH].cpu().numpy())
                        vn += len(logits)
                    del Xb, yb
                del Xr
        return np.concatenate(probs), np.concatenate(ys).astype(int), vloss / vn

# ---------- Bucle de 2 epocas completas (sin early stopping) ----------
for ep in range(1, EPOCHS + 1):
    t_ep = time.time()
    train_loss, n_tr, steps, tr_m = train_epoch()
    probs, ytrue, v_loss = val_epoch()
    val_probs_last, val_true_last = probs, ytrue
    vm = _metrics_from_scores(ytrue, probs, THRESHOLD)
    ckpt = {"epoch": ep, "state_dict": base_model.state_dict(),
            "train_loss": train_loss, "val_loss": v_loss, "seed": SEED,
            "config": {"input_dim": 2387, "arch": "2387-512-256-128-1",
                       "activation": "ReLU", "dropout": [0.3, 0.2, 0.1],
                       "loss": "BCEWithLogitsLoss", "threshold": THRESHOLD}}
    torch.save(ckpt, os.path.join(WORK, f"shadow_net_sorel_7m_v1.1_v5_epoch{ep}.pth"))
    torch.save(ckpt, CKPT_FINAL)
    improved = v_loss < best_val
    if improved:
        best_val = v_loss
        torch.save(ckpt, CKPT_BEST)
    row = {"epoch": ep,
           "train_loss": round(train_loss, 6),
           "train_accuracy": round(tr_m["accuracy"], 6),
           "train_precision": round(tr_m["precision"], 6),
           "train_recall": round(tr_m["recall"], 6),
           "train_f1": round(tr_m["f1"], 6),
           "train_tn": tr_m["tn"], "train_fp": tr_m["fp"],
           "train_fn": tr_m["fn"], "train_tp": tr_m["tp"],
           "val_loss": round(v_loss, 6),
           "val_accuracy": round(vm["accuracy"], 6),
           "val_precision": round(vm["precision"], 6),
           "val_recall": round(vm["recall"], 6),
           "val_f1": round(vm["f1"], 6),
           "val_roc_auc": round(vm["roc_auc"], 6),
           "val_pr_auc": round(vm["pr_auc"], 6),
           "tn": vm["tn"], "fp": vm["fp"], "fn": vm["fn"], "tp": vm["tp"],
           "steps": steps, "train_rows": n_tr, "val_rows": len(ytrue),
           "epoch_s": round(time.time() - t_ep, 1), "best": bool(improved)}
    hist_rows.append(row)
    print(f"ep{ep}: train_loss={train_loss:.4f} val_loss={v_loss:.4f} "
          f"val_acc={vm['accuracy']:.4f} val_f1={vm['f1']:.4f} "
          f"val_roc_auc={vm['roc_auc']:.4f} steps={steps} "
          f"rows={n_tr}/{len(ytrue)} ({time.time()-t_ep:.0f}s){' BEST' if improved else ''}",
          flush=True)

with open(HIST, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(hist_rows[0].keys()))
    w.writeheader()
    w.writerows(hist_rows)
print(f"fetch={t_fetch:.0f}s proc={t_proc:.0f}s cache_write={t_write:.0f}s "
      f"train={t_train:.0f}s val={t_val:.0f}s GB_leidos={n_fetch_bytes/2**30:.1f}")
print(f"PASS Fase 5B: {len(hist_rows)} epocas completas ({time.time()-T5B:.0f}s), best_val_loss={best_val:.4f}.")


In [ ]:
# FASE 5C (v5) — Evaluacion final (final y best) desde cache, metricas, metadata, hashes
# y validacion de recarga. Threshold fijo 0.5, sin tuning. Sin tocar produccion.
import time, hashlib, json
import numpy as np, torch
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
T5C = time.time()

def _eval_ckpt(path):
    ck = torch.load(path, map_location=DEV, weights_only=False)
    base_model.load_state_dict(ck["state_dict"])
    base_model.eval()
    if USE_CACHE:
        probs = np.empty(len(val_slots), dtype=np.float32)
        w = 0
        with torch.no_grad():
            for a in range(0, len(val_slots), BATCH):
                sl = val_slots[a:a + BATCH]
                xb = torch.from_numpy(np.array(X_CACHE[sl])).to(DEV)
                p = torch.sigmoid(base_model(xb)).reshape(-1).float().cpu().numpy()
                probs[w:w + len(p)] = p
                w += len(p)
                del xb
        y = np.array(Y_CACHE[val_slots]).astype(int)
    else:
        probs, y, _ = val_epoch()
    pred = (probs >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()
    m = {"n_val": int(len(y)), "threshold": THRESHOLD,
         "accuracy": round(float(accuracy_score(y, pred)), 6),
         "precision": round(float(precision_score(y, pred, zero_division=0)), 6),
         "recall": round(float(recall_score(y, pred, zero_division=0)), 6),
         "f1": round(float(f1_score(y, pred, zero_division=0)), 6),
         "roc_auc": round(float(roc_auc_score(y, probs)), 6),
         "pr_auc": round(float(average_precision_score(y, probs)), 6),
         "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
         "score_p50_all": round(float(np.median(probs)), 6),
         "score_mean_mal": round(float(probs[y == 1].mean()), 6),
         "score_mean_ben": round(float(probs[y == 0].mean()), 6)}
    return ck, m, probs, y

best_ck, best_metrics, best_probs, Y = _eval_ckpt(CKPT_BEST)
final_ck, final_metrics, P_final, Yf = _eval_ckpt(CKPT_FINAL)
assert np.array_equal(Y, Yf)
print(f"final ckpt: epoch={final_ck['epoch']} val_loss={final_ck['val_loss']:.4f}")
print(f"best  ckpt: epoch={best_ck['epoch']} val_loss={best_ck['val_loss']:.4f}")
print(json.dumps(final_metrics, indent=1))
print("threshold=0.5 reportado sin busqueda (no se asume optimo).")
np.save(os.path.join(WORK, "shadow_net_sorel_7m_v1.1_v5_val_probs.npy"), P_final)
np.save(os.path.join(WORK, "shadow_net_sorel_7m_v1.1_v5_val_true.npy"), Y)

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

final_epoch = int(final_ck["epoch"])
best_epoch = int(best_ck["epoch"])
metadata = {
    "model_version": "shadow_net_sorel_7m_v1.1",
    "experiment_version": "fase5_v5",
    "dataset": "SOREL-20M train split oficial (seleccion 7M seed 42)",
    "dataset_size": 7_000_000,
    "train_size": 6_300_000,
    "validation_size": 700_000,
    "seed": SEED,
    "input_dim": 2387,
    "ember_features": 2381,
    "overlay_features": ["slack_ratio", "slack_bytes_log", "file_size_log",
                         "imports_log", "has_cert", "stub_overlay_pattern"],
    "architecture": "2387->512->256->128->1",
    "hidden_layers": [512, 256, 128],
    "activation": "ReLU",
    "dropout": [0.3, 0.2, 0.1],
    "batch_norm": True,
    "batch_size": 8192,
    "epochs": EPOCHS,
    "learning_rate": 1e-3,
    "optimizer": "Adam",
    "loss_function": "BCEWithLogitsLoss",
    "threshold": THRESHOLD,
    "device": str(DEV),
    "n_gpu": N_GPU,
    "gpu_name": [g["name"] for g in GPU_DESC] if GPU_DESC else None,
    "gpu_capability": [g["capability"] for g in GPU_DESC] if GPU_DESC else None,
    "amp": bool(USE_AMP),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "training_time": round(t_train, 1),
    "fetch_time": round(t_fetch, 1),
    "processing_time": round(t_proc, 1),
    "cache_write_time": round(t_write, 1),
    "validation_time": round(t_val, 1),
    "cache_used": bool(USE_CACHE),
    "best_epoch": best_epoch,
    "final_epoch": final_epoch,
    "train_metrics": hist_rows,
    "validation_metrics": final_metrics,
    "best_validation_metrics": best_metrics,
    "feature_contract": "ShadowNetFeatures_v1.1 = EMBER_2381 + OVERLAY_6",
    "scaler_ember_filename": "scaler_sorel_7m_v1.1.pkl",
    "scaler_ember_sha256": EXP_EMBER,
    "scaler_overlay_filename": "scaler_overlay_sorel_7m_v1.1.pkl",
    "scaler_overlay_sha256": EXP_OVER,
    "split": {"method": "temporal por rl_fs_t (estable, sin RNG)", "train": 6_300_000,
              "val": 700_000, "corte_ts": float(t_bound),
              "nota": "val dentro del rango TRAIN; test futuro real = SOREL test"},
    "onnx_input_shape": ["batch", 2387],
    "onnx_output_shape": ["batch", 1],
    "onnx_output_type": "logits",
    "onnx_status": "READY_FOR_EXPORT",
    "timestamp_utc": __import__("datetime").datetime.now(
        __import__("datetime").timezone.utc).isoformat(),
    "versions": {"numpy": np.__version__, "pandas": pd.__version__,
                 "sklearn": __import__("sklearn").__version__,
                 "python": __import__("sys").version.split()[0]},
}
METRICS_JSON = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_v5_metrics.json")
METADATA_JSON = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_v5_metadata.json")
json.dump({"final": final_metrics, "best": best_metrics,
           "history": hist_rows}, open(METRICS_JSON, "w"), indent=1)
json.dump(metadata, open(METADATA_JSON, "w"), indent=1)

# --- validacion de recarga desde cero (instancia nueva del modelo) ---
fresh = ShadowNetMLP(2387).to(DEV)
fresh.load_state_dict(torch.load(CKPT_BEST, map_location=DEV, weights_only=False)["state_dict"])
fresh.eval()
with torch.no_grad():
    _probe = fresh(torch.zeros(4, 2387, device=DEV))
assert _probe.shape == (4, 1), _probe.shape
assert int(fresh.net[0].in_features) == 2387
print(f"reload test: PASS | output shape={tuple(_probe.shape)} input_dim=2387")

# --- hashes y puerta ---
for p in (CKPT_BEST, CKPT_FINAL, HIST, METRICS_JSON, METADATA_JSON):
    assert os.path.exists(p), f"FAIL: falta {p}"
    print(f"{os.path.basename(p)}: {os.path.getsize(p)/2**20:.2f}MB sha256={_sha(p)[:16]}...")
assert not any(os.path.exists(os.path.join(WORK, n)) for n in ("best_model.onnx", "scaler.pkl")), \
    "FAIL: colision con produccion"
print(f"TRAINING 7M: PASS ({time.time()-T5C:.0f}s eval+artefactos). "
      f"DETENER: sin ONNX, sin cambios en produccion.")


In [ ]:
# FASE 5D (v5) — Genero las 10 graficas del experimento actual (evidencia reproducible).
# Todas usan unicamente los resultados reales de este entrenamiento; no comparo con otros modelos.
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

DPI = 150
ep_axis = [r["epoch"] for r in hist_rows]
train_loss_ax = [r["train_loss"] for r in hist_rows]
val_loss_ax = [r["val_loss"] for r in hist_rows]
train_acc_ax = [r["train_accuracy"] for r in hist_rows]
val_acc_ax = [r["val_accuracy"] for r in hist_rows]
val_prec_ax = [r["val_precision"] for r in hist_rows]
val_rec_ax = [r["val_recall"] for r in hist_rows]
val_f1_ax = [r["val_f1"] for r in hist_rows]
val_roc_ax = [r["val_roc_auc"] for r in hist_rows]
val_pr_ax = [r["val_pr_auc"] for r in hist_rows]
tn_ax = [r["tn"] for r in hist_rows]
fp_ax = [r["fp"] for r in hist_rows]
fn_ax = [r["fn"] for r in hist_rows]
tp_ax = [r["tp"] for r in hist_rows]
plots_made = []

# 1) Curvas de perdida
plt.figure(figsize=(7, 4.5))
plt.plot(ep_axis, train_loss_ax, marker="o", label="Train loss")
plt.plot(ep_axis, val_loss_ax, marker="s", label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.xticks(ep_axis)
plt.title("Shadow-Net Defender — Training and Validation Loss")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_curves.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 2) Validation accuracy
plt.figure(figsize=(7, 4.5))
plt.plot(ep_axis, val_acc_ax, marker="o", color="tab:green", label="Validation accuracy")
plt.plot(ep_axis, train_acc_ax, marker="x", linestyle="--", color="tab:gray", label="Train accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.xticks(ep_axis)
plt.title("Validation Accuracy per Epoch"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_val_acc.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 3) Matriz de confusion (modelo final, epoca 2)
cm = np.array([[final_metrics["tn"], final_metrics["fp"]],
               [final_metrics["fn"], final_metrics["tp"]]])
plt.figure(figsize=(5.5, 4.8))
im = plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=11)
plt.xticks([0, 1], ["Pred Benign", "Pred Malware"])
plt.yticks([0, 1], ["Actual Benign", "Actual Malware"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix — Final Epoch"); plt.colorbar(im, fraction=0.046); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_confusion.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 4) ROC curve
fpr, tpr, _ = roc_curve(Y, P_final)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="tab:blue", label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_roc.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 5) Precision-Recall curve
prec, rec, _ = precision_recall_curve(Y, P_final)
pr_auc = average_precision_score(Y, P_final)
plt.figure(figsize=(6, 5))
plt.plot(rec, prec, color="tab:purple", label=f"PR-AUC = {pr_auc:.4f}")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_pr.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 6) Distribucion de scores
plt.figure(figsize=(7, 4.5))
plt.hist(P_final[Y == 0], bins=60, alpha=0.6, label="Benign", color="tab:green", density=True)
plt.hist(P_final[Y == 1], bins=60, alpha=0.6, label="Malware", color="tab:red", density=True)
plt.axvline(THRESHOLD, color="black", linestyle="--", label=f"threshold = {THRESHOLD:.2f}")
plt.xlabel("Score (sigmoid)"); plt.ylabel("Density")
plt.title("Score Distribution (Validation)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_scores.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 7) Throughput del pipeline
perf_labels = ["Fetch", "Processing", "Training"]
perf_vals = [t_fetch, t_proc, t_train]
plt.figure(figsize=(7, 4.5))
bars = plt.bar(perf_labels, perf_vals, color=["tab:orange", "tab:cyan", "tab:blue"])
for b, v in zip(bars, perf_vals):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:.0f}s", ha="center", va="bottom")
plt.xlabel("Stage"); plt.ylabel("Seconds")
plt.title("Pipeline Throughput (single cache pass + 2 epochs)")
plt.grid(alpha=0.3, axis="y"); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_perf.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 8) Precision / Recall / F1 por epoca (validacion)
x = np.arange(len(ep_axis)); wbar = 0.25
plt.figure(figsize=(7, 4.5))
plt.bar(x - wbar, val_prec_ax, wbar, label="Precision")
plt.bar(x, val_rec_ax, wbar, label="Recall")
plt.bar(x + wbar, val_f1_ax, wbar, label="F1")
plt.xticks(x, [f"ep{e}" for e in ep_axis]); plt.ylim(0, 1.05)
plt.xlabel("Epoch"); plt.ylabel("Score")
plt.title("Validation Precision / Recall / F1 per Epoch")
plt.legend(); plt.grid(alpha=0.3, axis="y"); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_prf1.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 9) ROC-AUC / PR-AUC por epoca
plt.figure(figsize=(7, 4.5))
plt.plot(ep_axis, val_roc_ax, marker="o", label="ROC-AUC")
plt.plot(ep_axis, val_pr_ax, marker="s", label="PR-AUC")
plt.xlabel("Epoch"); plt.ylabel("AUC"); plt.xticks(ep_axis); plt.ylim(0, 1.02)
plt.title("ROC-AUC / PR-AUC per Epoch")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_auc_epochs.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

# 10) TN / FP / FN / TP por epoca
x = np.arange(len(ep_axis)); wbar = 0.2
plt.figure(figsize=(8, 4.8))
plt.bar(x - 1.5 * wbar, tn_ax, wbar, label="TN")
plt.bar(x - 0.5 * wbar, fp_ax, wbar, label="FP")
plt.bar(x + 0.5 * wbar, fn_ax, wbar, label="FN")
plt.bar(x + 1.5 * wbar, tp_ax, wbar, label="TP")
plt.xticks(x, [f"ep{e}" for e in ep_axis])
plt.xlabel("Epoch"); plt.ylabel("Count (absolute)")
plt.title("Confusion Counts (TN/FP/FN/TP) per Epoch")
plt.legend(); plt.grid(alpha=0.3, axis="y"); plt.tight_layout()
p = os.path.join(WORK, "shadow_net_sorel_7m_v1.1_confusion_epochs.png"); plt.savefig(p, dpi=DPI); plt.close(); plots_made.append(p)

assert len(plots_made) == 10, f"FAIL: se generaron {len(plots_made)} graficas"
for p in plots_made:
    assert os.path.exists(p) and os.path.getsize(p) > 0
print(f"PLOTS: {len(plots_made)}/10 generadas.")


In [ ]:
# FASE 5 — Reporte final (v5). Calculo todos los valores a partir de los resultados reales.
_gpu = GPU_DESC[0]["name"] if GPU_DESC else "CPU"
_cap = GPU_DESC[0]["capability"] if GPU_DESC else "n/a"
_lines = [
    "=" * 60,
    "SHADOW-NET DEFENDER — FASE 5 v5",
    "=" * 60,
    "",
    "Dataset:",
    "  SOREL-20M",
    f"  Samples: {7_000_000:,}",
    "",
    "Architecture:",
    "  2387 -> 512 -> 256 -> 128 -> 1",
    "  MLP / Deep Learning",
    "",
    "Training:",
    f"  Epochs: {EPOCHS}",
    f"  Batch: {BATCH}",
    "  LR: 1e-3",
    "  Optimizer: Adam",
    "  Loss: BCEWithLogitsLoss",
    f"  Threshold: {THRESHOLD:.2f}",
    "",
    "Hardware:",
    f"  GPU: {_gpu} (cap {_cap}, n_gpu={N_GPU})",
    f"  CUDA: {torch.version.cuda if torch.cuda.is_available() else 'n/a'}",
    f"  PyTorch: {torch.__version__}",
    f"  AMP: {USE_AMP}",
    "",
    "Validation:",
    f"  Accuracy:  {final_metrics['accuracy']:.4f}",
    f"  Precision: {final_metrics['precision']:.4f}",
    f"  Recall:    {final_metrics['recall']:.4f}",
    f"  F1:        {final_metrics['f1']:.4f}",
    f"  ROC-AUC:   {final_metrics['roc_auc']:.4f}",
    f"  PR-AUC:    {final_metrics['pr_auc']:.4f}",
    "",
    "Confusion Matrix:",
    f"  TN: {final_metrics['tn']:,}",
    f"  FP: {final_metrics['fp']:,}",
    f"  FN: {final_metrics['fn']:,}",
    f"  TP: {final_metrics['tp']:,}",
    "",
    "Performance:",
    f"  Fetch: {t_fetch:.0f}s",
    f"  Processing: {t_proc:.0f}s",
    f"  Training: {t_train:.0f}s",
    "",
    "Artifacts:",
    "  model: shadow_net_sorel_7m_v1.1_v5_best.pth",
    "  model: shadow_net_sorel_7m_v1.1_v5_final.pth",
    "  history: shadow_net_sorel_7m_v1.1_v5_history.csv",
    "  metrics: shadow_net_sorel_7m_v1.1_v5_metrics.json",
    "  metadata: shadow_net_sorel_7m_v1.1_v5_metadata.json",
    "",
    "ONNX:",
    "  input_dim: 2387",
    "  output_dim: 1",
    "  status: READY_FOR_EXPORT",
    "",
    "Plots:",
    f"  {len(plots_made)}/10 generated",
    "",
    "=" * 60,
    "STATUS: PASS",
    "=" * 60,
]
print("\n".join(_lines))
